# WSB Top-200 - BMSB Buy-Readiness Screener

Applies the **Bull Market Support Band** strategy from `bmsb_strategy.ipynb` across the
**top 200 most-mentioned r/wallstreetbets tickers** (ApeWisdom), then **groups and ranks**
every name by how ready it is to buy *right now*.

For each ticker (weekly candles, 10y - the standard BMSB timeframe):
1. **Sweep** the same `W_GRID` parameter grid as the notebook.
2. Pick the **best config** with the notebook rule: keep 8-15 trade configs, rank by `Score = Total P&L x Win Rate`.
3. Evaluate the **live BUY/SELL/HOLD** signal on the latest bar.
4. Measure **proximity-to-trigger** and drop it into a bucket.

**Buckets**
- `BUY NOW` - fresh entry signal on the latest weekly bar (actionable).
- `CLOSE TO ENTERING` - flat but within ~6% of the trigger, or already in the anticipate buy-zone.
- `IN POSITION` - already trending above the band (strategy is holding); a new buyer is chasing.
- `SELL / EXIT` - strategy would be exiting.
- `WATCH` - flat and far from any trigger.

> **Research only, not financial advice.** Best-fit configs come from a parameter sweep and are partly curve-fit.

### First-time setup
```bash
pip install -r requirements.txt
```

In [ ]:
# ── PATH SETUP ──────────────────────────────────────────────────────────────
# Ensures the wsb_screener package is importable regardless of which directory
# VS Code / Jupyter starts the kernel in.
import sys, os

# In Jupyter, __file__ is not defined — fall back to CWD.
notebook_dir = os.getcwd()

# Walk upward (up to 5 levels) looking for the wsb_screener folder.
search = notebook_dir
found = False
for _ in range(6):
    if os.path.isdir(os.path.join(search, 'wsb_screener')):
        if search not in sys.path:
            sys.path.insert(0, search)
        os.chdir(search)        # keep results/ in the right place
        print(f'Repo root found at: {search}')
        found = True
        break
    parent = os.path.dirname(search)
    if parent == search:        # reached filesystem root
        break
    search = parent

if not found:
    print('WARNING: wsb_screener/ directory not found in any ancestor of the CWD.')
    print('Please open VS Code / Jupyter from inside the trading-demo repo folder,')
    print('or set the kernel working directory to that folder.')

print('Working directory:', os.getcwd())

In [ ]:
# ================= CONFIG - edit me =================
TOP_N     = 200        # how many WSB tickers to screen
PERIOD    = "10y"      # history length
INTERVAL  = "1wk"      # weekly = standard BMSB timeframe
NEAR_PCT  = 6.0        # within this % of the trigger => "close to entering"
OUTDIR    = "results"

# Weekly sweep grid - identical to W_GRID in bmsb_strategy.ipynb
#   (SMA lens, EMA lens, entry modes, RSI-buy thresholds, stop %, confirm bars)
W_GRID = ([15,20,25], [21,30,40], ["anticipate","breakout","reclaim"], [40,45,50], [0.08,0.12], [4,6])

In [ ]:
import pandas as pd
from wsb_screener import screener as scr
from wsb_screener.data import get_wsb_tickers, download_history
from wsb_screener.screener import run_screener
from wsb_screener.report import build_markdown

# push notebook config into the screener module
scr.W_GRID   = W_GRID
scr.NEAR_PCT = NEAR_PCT
pd.set_option("display.max_rows", 250, "display.width", 240, "display.max_columns", 40)

## 1. Fetch the WSB universe + price history

In [ ]:
tickers = get_wsb_tickers(TOP_N)
metas   = {t["ticker"]: t for t in tickers}
print(f"Top {len(metas)} r/wallstreetbets tickers by mention count")
print(", ".join(list(metas)[:25]), "...")

hist = download_history(list(metas), period=PERIOD, interval=INTERVAL)
print(f"\nDownloaded price data for {len(hist)}/{len(metas)} tickers")

## 2. Screen everything (sweep -> best config -> live signal -> bucket)

In [ ]:
df = run_screener(hist, metas)
signal_date = df["signal_date"].dropna().mode().iloc[0]
print("Latest weekly bar:", signal_date)
df["group"].value_counts()

## 3. BUY NOW - fresh entry signals on the latest bar

In [ ]:
cols = ["group_rank","ticker","name","close","readiness","recommendation","mode",
        "band_bot","band_top","rsi","need_move_%","bt_trades","bt_winrate_%","bt_pnl_$","bt_score"]
df[df.group=="BUY NOW"][cols]

## 4. CLOSE TO ENTERING - watch these this week

In [ ]:
cols = ["group_rank","ticker","name","close","readiness","pos_vs_band","mode",
        "band_bot","band_top","rsi","need_move_%","note"]
df[df.group=="CLOSE TO ENTERING"][cols]

## 5. IN POSITION (riding) / SELL / WATCH

In [ ]:
pcols = ["group_rank","ticker","name","close","recommendation","mode","band_bot","band_top","rsi","need_move_%","bt_winrate_%","bt_score"]
display(df[df.group=="IN POSITION"][pcols].head(25))
display(df[df.group=="SELL / EXIT"][pcols])
wcols = ["group_rank","ticker","name","close","readiness","pos_vs_band","mode","band_bot","band_top","rsi","need_move_%"]
display(df[df.group=="WATCH"][wcols].head(25))

## 6. Save CSV + Markdown report

In [ ]:
import os
os.makedirs(OUTDIR, exist_ok=True)
df.to_csv(f"{OUTDIR}/wsb_bmsb_screen.csv", index=False)
with open(f"{OUTDIR}/wsb_bmsb_report.md", "w") as f:
    f.write(build_markdown(df, len(metas), signal_date))
print("wrote", f"{OUTDIR}/wsb_bmsb_screen.csv", "and", f"{OUTDIR}/wsb_bmsb_report.md")

## 7. (Optional) Drill into one ticker's BMSB chart
Reuses the exact backtest/plot from `bmsb_strategy.ipynb` for any symbol in the table.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from wsb_screener.bmsb_core import bmsb_backtest, compute_stats

SYMBOL = "NU"   # <- pick any ticker from the tables above
row = df[df.ticker==SYMBOL].iloc[0]
d, t = bmsb_backtest(hist[SYMBOL], int(row.sma), int(row.ema), row['mode'], int(row.rsi_buy), float(row.stop), int(row.cbars))

fig, ax = plt.subplots(figsize=(15,6))
ax.plot(d.Date, d.Close, color="#222", lw=1.1, label="Close")
ax.plot(d.Date, d.SMA, color="#1f77b4", lw=1.0, label=f"{int(row.sma)} SMA")
ax.plot(d.Date, d.EMA, color="#ff7f0e", lw=1.0, label=f"{int(row.ema)} EMA")
ax.fill_between(d.Date, d.band_bot, d.band_top, color="#2ca02c", alpha=0.15, label="Support Band")
if not t.empty:
    ax.scatter(t.entry_date, t.entry_price, marker="^", s=110, color="green", edgecolor="k", zorder=5, label="BUY")
    ax.scatter(t.exit_date,  t.exit_price,  marker="v", s=110, color="red",   edgecolor="k", zorder=5, label="SELL")
ax.set_title(f"{SYMBOL} - BMSB weekly (SMA{int(row.sma)}/EMA{int(row.ema)}, mode={row['mode']}) | bucket: {row.group}", weight="bold")
ax.legend(loc="upper left", ncol=2, fontsize=9); ax.grid(alpha=0.25)
plt.show()
print(compute_stats(t))